# PY-V — Kaggle runner

All big training and tests for V run here as a **background run** on Kaggle's **GPU T4 x2** — the laptop can be off.
One button: the code cell below sets itself up (code from GitHub, packages) and runs `scripts/gpu_pipeline.py`.
Both GPUs work at once (Granite chat on GPU 0, Granite plain on GPU 1) while the training data is made on the CPU.

**One-time setup**
1. Kaggle → Settings → verify your phone number (needed for GPU and internet).
2. Datasets → New Dataset → upload the training source files: the `.jsonl` files from Drive `MyDrive/PY-V/results/data_v2`
   (not `_sample` / `.part`) + the laptop file `data/raw/v2/old_github.jsonl`. Keep it **Private**; any name.
3. Create → New Notebook → File → Import Notebook → this file.

**Each run**
1. Push your latest commit to GitHub — the notebook downloads the code from there.
2. Session options (right panel): Accelerator **GPU T4 x2**, Internet **on**.
3. Add Input → your data set.
4. **Save Version → Save & Run All (Commit) → Save.** Then close the tab — it runs on Kaggle (~5 h).
   It stops its jobs by itself after 11 h (Kaggle's limit is 12 h a run), so everything done is always saved.

**Continuing a stopped run**: Add Input → Notebook Output → this notebook (latest version), then Save & Run All again.
Finished stages are skipped; training continues from its last checkpoint.

**Results** — the version's Output tab, folder `PY-V/`: `results/PIPELINE_REPORT.md` (scores, stages, training),
`results/pipeline_log.txt` (full log), `results/eval/` (every answer), `model_<brain>/lora/` (trained adapters).
On the laptop: `kaggle kernels output <user>/<notebook> -p "Kaggle downloads"` (needs the Kaggle key in `~/.kaggle/`).

Kaggle and Colab keep separate results (Kaggle output vs Drive) — finish a job where it started.
Colab is the backup: `Google Colab/py_v_runner.ipynb`, same pipeline on one T4.

In [ ]:
# ▶ RUN EVERYTHING — setup + every big job (safe to re-run: finished stages are skipped)
import os

REPO_URL = "https://github.com/Alexie1171/Py-V.git"
REPO_DIR = "/tmp/Py-V"               # the code — not part of the saved output
ROOT     = "/kaggle/working/PY-V"    # results + trained adapters — saved as this run's output

if not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    raise SystemExit("This notebook is for Kaggle - on Colab use Google Colab/py_v_runner.ipynb")
!nvidia-smi -L

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
if not os.path.isfile(f"{REPO_DIR}/scripts/gpu_pipeline.py"):
    raise SystemExit("Code missing - is Internet on (Session options)? Is the latest commit pushed to GitHub?")
%cd {REPO_DIR}
!git log -1 --format="code version: %h  %s  (%ad)" --date=short
!pip install -q -U peft bitsandbytes

print("Inputs:")
!find /kaggle/input -maxdepth 3 | head -40
!python -m scripts.gpu_pipeline --root {ROOT} --inputs /kaggle/input --stop-after 11